# 📓 Notebook 04 — Basic RAG (No Vector DB)---**Phase 2 · Retrieval-Augmented Generation**> RAG is the most important pattern in production LLM systems. It lets models answer questions using YOUR data.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| RAG architecture | The retrieve → augment → generate pipeline || Chunking strategies | How to split documents for optimal retrieval || Embedding + retrieval | Convert chunks to vectors, find relevant ones || Context injection | Feed retrieved context to the LLM prompt || RAG vs fine-tuning | When to use which (interview-critical) |### 🏗️ Build: RAG Pipeline Over PDF/TXT Documents

## 1. Setup

In [ ]:
import os, json, timeimport numpy as npfrom dotenv import load_dotenvimport litellmload_dotenv()DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-4o-mini")DEFAULT_EMBEDDING = os.getenv("DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small")def get_embedding(text, model=None):    model = model or DEFAULT_EMBEDDING    response = litellm.embedding(model=model, input=[text])    return response.data[0]["embedding"]def get_embeddings_batch(texts, model=None):    model = model or DEFAULT_EMBEDDING    response = litellm.embedding(model=model, input=texts)    return [d["embedding"] for d in response.data]def cosine_similarity(a, b):    a, b = np.array(a), np.array(b)    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))print(f"🤖 LLM: {DEFAULT_MODEL} | 📐 Embeddings: {DEFAULT_EMBEDDING}")

---## 2. The RAG Architecture### Why RAG?LLMs know what they were trained on, but not YOUR data — your documents, your database, your knowledge base.```        ┌──────────────┐        │  User Query  │        └──────┬───────┘               ↓     ┌─────────────────────┐     │  1. RETRIEVE         │ ← Find relevant chunks from your data     │     (Embedding Search)│     └─────────┬───────────┘               ↓     ┌─────────────────────┐     │  2. AUGMENT          │ ← Inject retrieved context into prompt     │     (Prompt Building) │     └─────────┬───────────┘               ↓     ┌─────────────────────┐     │  3. GENERATE         │ ← LLM generates answer using context     │     (LLM Call)        │     └─────────────────────┘```### RAG vs Fine-Tuning (Interview Critical)| Aspect | RAG | Fine-Tuning ||--------|-----|-------------|| Data freshness | Real-time (swap documents anytime) | Static (retrain needed) || Setup cost | Low (embed docs + prompt) | High (training data + GPU hours) || Accuracy | Good (with good retrieval) | Better (for narrow domains) || Hallucination | Lower (grounded in retrieved text) | Can still hallucinate || Scalability | Easy to add new docs | Need to retrain || Best for | Dynamic knowledge bases, many topics | Specialized behavior, style, format |> **Interview Tip:** *"When would you use RAG vs fine-tuning?"*  > RAG for dynamic data and broad knowledge. Fine-tuning for behavioral changes and specialized formats.  > Best practice: Start with RAG. Only fine-tune if RAG quality is insufficient for your specific task.

---## 3. Document Chunking Strategies### Why Chunk?- LLMs have context window limits (4K-128K tokens)- Embedding models have token limits (usually 8K)- Smaller chunks = more precise retrieval- But too small = loss of context### The Three Main Strategies

In [ ]:
import re# ===== Strategy 1: Fixed-Size Chunking =====def chunk_fixed(text, chunk_size=500, overlap=50):    """Split text into fixed-size chunks with overlap.    Simple, fast, but may split mid-sentence.    """    chunks = []    start = 0    while start < len(text):        end = start + chunk_size        chunk = text[start:end]        chunks.append({            "text": chunk.strip(),            "start": start,            "end": min(end, len(text)),            "method": "fixed"        })        start += chunk_size - overlap    return [c for c in chunks if c["text"]]# ===== Strategy 2: Recursive Character Splitting =====def chunk_recursive(text, chunk_size=500, overlap=50, separators=None):    """Split by progressively finer separators.    Tries to split on paragraphs, then sentences, then words.    """    if separators is None:        separators = ["\n\n", "\n", ". ", " "]        chunks = []        def _split(text, sep_idx=0):        if len(text) <= chunk_size:            if text.strip():                chunks.append({"text": text.strip(), "method": "recursive"})            return                if sep_idx >= len(separators):            # Last resort: hard split            chunks.append({"text": text[:chunk_size].strip(), "method": "recursive"})            _split(text[chunk_size - overlap:], sep_idx)            return                sep = separators[sep_idx]        parts = text.split(sep)                current = ""        for part in parts:            if len(current) + len(part) + len(sep) <= chunk_size:                current += part + sep            else:                if current.strip():                    chunks.append({"text": current.strip(), "method": "recursive"})                current = part + sep        if current.strip():            if len(current) > chunk_size:                _split(current, sep_idx + 1)            else:                chunks.append({"text": current.strip(), "method": "recursive"})        _split(text)    return chunks# ===== Strategy 3: Sentence-Based Chunking =====def chunk_sentences(text, max_sentences=5, overlap_sentences=1):    """Split into groups of N sentences.    Best for preserving complete thoughts.    """    sentences = re.split(r'(?<=[.!?])\s+', text)    sentences = [s.strip() for s in sentences if s.strip()]        chunks = []    start = 0    while start < len(sentences):        end = min(start + max_sentences, len(sentences))        chunk_text = " ".join(sentences[start:end])        chunks.append({            "text": chunk_text,            "sentences": sentences[start:end],            "method": "sentence"        })        start += max_sentences - overlap_sentences    return chunksprint("✅ Three chunking strategies defined")

In [ ]:
# Compare strategies on a sample documentsample_document = """Artificial Intelligence (AI) has transformed numerous industries in the past decade. Machine learning, a subset of AI, enables computers to learn from data without explicit programming. Deep learning, using neural networks with many layers, has achieved breakthrough results in image recognition, natural language processing, and game playing.The key components of a modern AI system include data collection, preprocessing, model training, and deployment. Data quality is crucial — the saying "garbage in, garbage out" is especially true for AI. Clean, well-labeled data can make the difference between a model that works and one that doesn't.Transfer learning has revolutionized the field by allowing models trained on large datasets to be fine-tuned for specific tasks. This dramatically reduces the amount of training data and compute needed. Models like BERT, GPT, and T5 are pre-trained on massive text corpora and can be adapted for classification, generation, summarization, and more.Responsible AI is an emerging field focused on ensuring AI systems are fair, transparent, and accountable. This includes addressing bias in training data, providing explanations for model decisions, and establishing governance frameworks.The future of AI includes multi-modal models that can process text, images, audio, and video simultaneously. Large Language Models (LLMs) are becoming more efficient and capable, with applications ranging from coding assistants to scientific research."""print("📊 Chunking Strategy Comparison")print("=" * 70)for name, func, kwargs in [    ("Fixed (500 chars)", chunk_fixed, {"chunk_size": 500, "overlap": 50}),    ("Recursive", chunk_recursive, {"chunk_size": 500, "overlap": 50}),    ("Sentence (5 per chunk)", chunk_sentences, {"max_sentences": 5, "overlap_sentences": 1}),]:    chunks = func(sample_document, **kwargs)    print(f"\n🔹 {name}: {len(chunks)} chunks")    for i, c in enumerate(chunks):        text = c["text"][:80].replace("\n", " ")        print(f"   [{i}] ({len(c['text'])} chars) {text}...")

### Chunking Strategy Decision Matrix| Strategy | Pros | Cons | Best For ||----------|------|------|----------|| **Fixed-size** | Simple, predictable | May split mid-sentence | Uniform documents || **Recursive** | Respects structure | More complex logic | Most text documents || **Sentence** | Preserves meaning | Uneven chunk sizes | Articles, reports || **Semantic** | Best coherence | Slow (needs embeddings) | High-quality retrieval |> **Interview Tip:** *"What chunking strategy would you use?"*  > Recursive with overlap is the best default. Chunk size 200-500 tokens.  > Overlap 10-20% prevents information loss at boundaries.

---## 4. Building the RAG Pipeline

In [ ]:
class BasicRAG:    """A complete RAG pipeline — no vector DB, pure numpy."""        def __init__(self, model=None, embedding_model=None):        self.llm_model = model or DEFAULT_MODEL        self.emb_model = embedding_model or DEFAULT_EMBEDDING        self.chunks = []        self.embeddings = []        def ingest(self, text, chunk_strategy="recursive", **kwargs):        """Chunk and embed a document."""        # Step 1: Chunk        if chunk_strategy == "fixed":            self.chunks = chunk_fixed(text, **kwargs)        elif chunk_strategy == "recursive":            self.chunks = chunk_recursive(text, **kwargs)        elif chunk_strategy == "sentence":            self.chunks = chunk_sentences(text, **kwargs)                print(f"  📄 Chunked into {len(self.chunks)} pieces")                # Step 2: Embed all chunks        texts = [c["text"] for c in self.chunks]        self.embeddings = get_embeddings_batch(texts, model=self.emb_model)        print(f"  📐 Embedded {len(self.embeddings)} chunks ({len(self.embeddings[0])}D)")        def retrieve(self, query, top_k=3):        """Find most relevant chunks for a query."""        query_emb = get_embedding(query, model=self.emb_model)                scored = []        for i, doc_emb in enumerate(self.embeddings):            score = cosine_similarity(query_emb, doc_emb)            scored.append((i, score))                scored.sort(key=lambda x: x[1], reverse=True)                results = []        for idx, score in scored[:top_k]:            results.append({                "chunk_id": idx,                "text": self.chunks[idx]["text"],                "score": round(score, 4),            })        return results        def generate(self, query, context_chunks):        """Generate answer using retrieved context."""        context = "\n\n---\n\n".join([c["text"] for c in context_chunks])                prompt = f"""Answer the question based ONLY on the provided context. If the context doesn't contain enough information, say "I don't have enough information to answer this."Context:{context}Question: {query}Answer:"""                response = litellm.completion(            model=self.llm_model,            messages=[                {"role": "system", "content": "You are a helpful assistant that answers questions based on provided context. Be precise and cite relevant parts of the context."},                {"role": "user", "content": prompt}            ],            temperature=0,            max_tokens=500,        )        return response.choices[0].message.content        def query(self, question, top_k=3, verbose=True):        """Full RAG pipeline: retrieve → augment → generate."""        # Retrieve        chunks = self.retrieve(question, top_k=top_k)                if verbose:            print(f"\n🔍 Retrieved {len(chunks)} chunks:")            for c in chunks:                print(f"   [{c['score']:.4f}] {c['text'][:60]}...")                # Generate        answer = self.generate(question, chunks)                if verbose:            print(f"\n📝 Answer: {answer}")                return {"answer": answer, "chunks": chunks}print("✅ BasicRAG pipeline ready")

In [ ]:
# Build and test the RAG pipelinerag = BasicRAG()# Ingest the sample documentprint("📥 Ingesting document...")rag.ingest(sample_document, chunk_strategy="recursive", chunk_size=300, overlap=30)# Ask questionsprint("\n" + "=" * 70)questions = [    "What is transfer learning and why is it important?",    "What are the key components of a modern AI system?",    "How does responsible AI address bias?",    "What is the future of AI according to this text?",]for q in questions:    print(f"\n❓ {q}")    result = rag.query(q, top_k=2, verbose=True)    print("-" * 70)

---## 5. Loading Real Documents

In [ ]:
# Reading a PDF (if available)try:    from pypdf import PdfReader        def load_pdf(path):        """Extract text from a PDF file."""        reader = PdfReader(path)        text = ""        for page in reader.pages:            text += page.extract_text() + "\n\n"        return text.strip()        print("✅ PDF loading ready — use: text = load_pdf('your_file.pdf')")except ImportError:    print("⚠️ Install pypdf: pip install pypdf")# Reading a TXT filedef load_txt(path):    """Load text from a file."""    with open(path, "r", encoding="utf-8") as f:        return f.read()# Example: Create a sample knowledge base file and RAG over itsample_kb = """# Company Handbook - TechCorp Inc.## Leave PolicyEmployees are entitled to 20 days of paid leave per year. Sick leave is separate and unlimited with a doctor's note. Maternity leave is 6 months and paternity leave is 4 weeks. Leave can be carried forward up to 5 days to the next year.## Remote Work PolicyEmployees may work remotely up to 3 days per week. Full remote requires manager approval. All remote employees must be available during core hours (10am-4pm local time). VPN must be used for accessing company systems remotely.## Expense PolicyBusiness travel must be pre-approved. Economy class flights for trips under 5 hours. Hotel budget is $200/night domestic, $300/night international. Meals are reimbursed up to $60/day. Receipts are required for all expenses over $25.## Performance ReviewsAnnual reviews happen in Q4. 360-degree feedback is collected from peers, managers, and direct reports. Ratings are on a 1-5 scale. Bonus is tied to both individual and company performance. Top performers (rating 4+) are eligible for stock grants.## IT SecurityAll devices must have endpoint protection. Passwords must be 12+ characters with MFA enabled. Don't share credentials. Report suspicious emails to security@techcorp.com. USB drives are not permitted on company devices."""# Save and RAG over itwith open(os.path.join(os.environ.get('TEMP', '/tmp'), "sample_handbook.txt"), "w") as f:    f.write(sample_kb)print("📄 Sample handbook created")print("\n🔨 Building RAG pipeline over handbook...")handbook_rag = BasicRAG()handbook_rag.ingest(sample_kb, chunk_strategy="recursive", chunk_size=300, overlap=30)# Test HR questionshr_questions = [    "How many vacation days do I get?",    "Can I work from home every day?",    "What's the hotel budget for international travel?",    "How are bonuses calculated?",    "What's the password policy?",]print("\n" + "=" * 70)print("📋 HR Chatbot Demo")for q in hr_questions:    print(f"\n❓ {q}")    result = handbook_rag.query(q, top_k=2)    print("-" * 50)

---## 📝 Interview Quiz — RAG Fundamentals### Q1: Explain the RAG architecture. What are the three steps?<details><summary>💡 Answer</summary>1. **Retrieve** — Embed the user query, search the document index for most similar chunks (cosine similarity)2. **Augment** — Inject retrieved chunks into the LLM prompt as context3. **Generate** — LLM generates an answer grounded in the provided contextKey: The LLM never "sees" the entire document corpus. It only receives the top-k most relevant chunks. This is why retrieval quality directly determines answer quality.</details>### Q2: What is the ideal chunk size? How does overlap help?<details><summary>💡 Answer</summary>**Chunk size:** 200-500 tokens is typical. Trade-off:- **Too large** → Less precise retrieval, wastes context window- **Too small** → Loses context, fragments ideas**Overlap** (10-20% of chunk size):- Prevents information loss at chunk boundaries- If a key fact spans two chunks, overlap ensures at least one chunk captures it fully- Trade-off: More chunks = more storage + slightly slower search</details>### Q3: RAG vs Fine-Tuning — when would you choose each?<details><summary>💡 Answer</summary>| Choose RAG when... | Choose Fine-Tuning when... ||-|-|| Data changes frequently | Need specialized behavior/format || Many documents/topics | Narrow domain, stable data || Need source attribution | Need faster inference || Quick setup required | Have labeled training data + GPU budget || Don't want to retrain models | RAG retrieval quality is insufficient |**Best practice:** Start with RAG always. It's faster to set up and easier to maintain. Only fine-tune if you've exhausted RAG optimization and still need better quality.</details>### Q4: What happens when the retrieved chunks don't contain the answer?<details><summary>💡 Answer</summary>The LLM may:- **Hallucinate** an answer using its training data (dangerous!)- **Say "I don't know"** if properly instructed in the system prompt**Defenses:**1. System prompt: "Answer ONLY from the provided context"2. Confidence threshold: If max retrieval score < 0.3, say "not found"3. Abstain instruction: "If uncertain, say 'I need more information'"4. Source citation: Force the LLM to quote the specific context it used</details>### Q5: How would you evaluate a RAG system?<details><summary>💡 Answer</summary>Three dimensions (we'll cover this in depth in Notebook 15):1. **Retrieval quality** — Are the right chunks being retrieved? (Precision@K, Recall@K)2. **Answer faithfulness** — Is the answer supported by the retrieved context? (No hallucination)3. **Answer relevance** — Does the answer actually address the question?Metrics: RAGAS framework, human evaluation, LLM-as-judge for automated scoring.</details>

---## ✅ Notebook 04 Summary| Concept | Key Takeaway ||---------|-------------|| RAG | Retrieve → Augment → Generate; grounds LLMs in your data || Chunking | Recursive with overlap is best default; 200-500 tokens || Retrieval | Cosine similarity between query and chunk embeddings || Context injection | Place retrieved chunks in prompt before the question || RAG vs Fine-tuning | RAG for dynamic data; fine-tuning for specialized behavior || Hallucination defense | System prompt + confidence threshold + citation |### ➡️ Next: [Notebook 05 — Advanced RAG](./05_advanced_rag.ipynb)*Hybrid search, reranking, and production-grade retrieval optimization.*